In [1]:
import base64
from collections import Counter

In [2]:
def hamming_distance(a, b):
    return sum(bin(x ^ y).count("1") for x, y in zip(a, b))

In [3]:
ENGLISH_FREQ = {
    'a': 8.2, 'b': 1.5, 'c': 2.8, 'd': 4.3,
    'e': 12.7, 'f': 2.2, 'g': 2.0, 'h': 6.1,
    'i': 7.0, 'j': 0.15, 'k': 0.77, 'l': 4.0,
    'm': 2.4, 'n': 6.7, 'o': 7.5, 'p': 1.9,
    'q': 0.095, 'r': 6.0, 's': 6.3, 't': 9.1,
    'u': 2.8, 'v': 0.98, 'w': 2.4, 'x': 0.15,
    'y': 2.0, 'z': 0.074, ' ': 13.0
}

In [4]:
def english_score(text):
    score = 0

    for byte in text:
        c = chr(byte).lower()

        if c in ENGLISH_FREQ:
            score += ENGLISH_FREQ[c]
        elif 32 <= byte <= 126:
            score += 0
        else:
            score -= 50

    return score

In [5]:
def break_single_byte_xor(ciphertext):
    best_score = float('-inf')
    best_key = None
    best_plaintext = None

    for key in range(256):
        plaintext = bytes(c ^ key for c in ciphertext)

        score = english_score(plaintext)

        if score > best_score:
            best_score = score
            best_key = key
            best_plaintext = plaintext

    return best_key, best_plaintext


In [6]:
s1 = b"this is a test"
s2 = b"wokka wokka!!!"

print("Expected: 37")
print("Computed:", hamming_distance(s1, s2))
print()

Expected: 37
Computed: 37



In [7]:
def guess_keysizes(ciphertext):

    candidates = []

    for keysize in range(2, 41):

        chunks = [
            ciphertext[i:i + keysize]
            for i in range(0, keysize * 8, keysize)
        ]

        if len(chunks) < 8:
            continue

        distances = []

        for i in range(len(chunks) - 1):

            distance = hamming_distance(
                chunks[i],
                chunks[i + 1]
            )

            distances.append(distance / keysize)

        avg_distance = sum(distances) / len(distances)

        candidates.append((avg_distance, keysize))

    candidates.sort()

    return [k for _, k in candidates[:5]]


In [8]:
def break_repeating_key_xor(ciphertext):

    candidate_keysizes = guess_keysizes(ciphertext)

    best_score = float("-inf")
    best_key = b""
    best_plaintext = b""

    for keysize in candidate_keysizes:

        blocks = [
            ciphertext[i:i + keysize]
            for i in range(0, len(ciphertext), keysize)
        ]

        transposed = []

        for i in range(keysize):

            column = bytearray()

            for block in blocks:

                if i < len(block):
                    column.append(block[i])

            transposed.append(bytes(column))

        key = bytearray()

        for block in transposed:

            key_byte, _ = break_single_byte_xor(block)

            key.append(key_byte)

        plaintext = bytes(
            ciphertext[i] ^ key[i % len(key)]
            for i in range(len(ciphertext))
        )

        score = english_score(plaintext)

        if score > best_score:
            best_score = score
            best_key = bytes(key)
            best_plaintext = plaintext

    return best_key, best_plaintext

In [9]:
with open("6.txt", "r") as f:
    ciphertext = base64.b64decode(f.read())


In [10]:
key, plaintext = break_repeating_key_xor(ciphertext)

In [11]:
print("=" * 60)
print("RECOVERED KEY")
print("=" * 60)
print(key.decode(errors="replace"))

print()

print("=" * 60)
print("PLAINTEXT PREVIEW")
print("=" * 60)

print(
    plaintext[:3000].decode(
        errors="replace"
    )
)

RECOVERED KEY
Terminator X: Bring the noise

PLAINTEXT PREVIEW
I'm back and I'm ringin' the bell 
A rockin' on the mike while the fly girls yell 
In ecstasy in the back of me 
Well that's my DJ Deshay cuttin' all them Z's 
Hittin' hard and the girlies goin' crazy 
Vanilla's on the mike, man I'm not lazy. 

I'm lettin' my drug kick in 
It controls my mouth and I begin 
To just let it flow, let my concepts go 
My posse's to the side yellin', Go Vanilla Go! 

Smooth 'cause that's the way I will be 
And if you don't give a damn, then 
Why you starin' at me 
So get off 'cause I control the stage 
There's no dissin' allowed 
I'm in my own phase 
The girlies sa y they love me and that is ok 
And I can dance better than any kid n' play 

Stage 2 -- Yea the one ya' wanna listen to 
It's off my head so let the beat play through 
So I can funk it up and make it sound good 
1-2-3 Yo -- Knock on some wood 
For good luck, I like my rhymes atrocious 
Supercalafragilisticexpialidocious 
I'm an effect 